# ПР-14. Сохранение модели, инференс и мониторинг

Подготовить воспроизводимый пайплайн к эксплуатации: сериализовать его целиком, восстановить, проверить схему входа и показать простую проверку дрейфа.

**Данные:** см. README проекта. Выполняйте ячейки сверху вниз; фиксируйте результаты в отчёте.

## 1. Подготовка данных

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent
data = pd.read_csv(ROOT / 'data' / 'classification_breast_cancer.csv')
X = data.drop(columns=['target', 'target_name'])
y = data['target'].astype(int)  # 0 = malignant, 1 = benign
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print('train/test:', X_train.shape, X_test.shape)


## 1. Обучение и сериализация полного пайплайна

Сохраняем вместе с моделью преобразование признаков. Это позволяет применять к новым данным ту же обработку, что и при обучении.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
import joblib, json, sklearn
model=Pipeline([('scale',StandardScaler()),('classifier',LogisticRegression(C=1,max_iter=3000,random_state=42))])
model.fit(X_train,y_train)
print('holdout F1 macro:',round(f1_score(y_test,model.predict(X_test),average='macro'),3))
artifacts=ROOT/'artifacts'; artifacts.mkdir(exist_ok=True)
joblib.dump(model,artifacts/'breast_cancer_pipeline.joblib')
metadata={'features':list(X.columns),'target_mapping':{'0':'malignant','1':'benign'},
          'sklearn_version':sklearn.__version__,'random_state':42}
(artifacts/'model_metadata.json').write_text(json.dumps(metadata,indent=2),encoding='utf-8')
print('Сохранено:',artifacts)

## 2. Загрузка и проверка инференса

Проверьте порядок и названия признаков. В реальном сервисе входной запрос нужно валидировать и возвращать понятную ошибку при неверной схеме.

In [ ]:
loaded=joblib.load(artifacts/'breast_cancer_pipeline.joblib')
meta=json.loads((artifacts/'model_metadata.json').read_text(encoding='utf-8'))
assert list(X_test.columns)==meta['features']
sample=X_test.iloc[[0]].copy()
pred=int(loaded.predict(sample)[0])
proba=loaded.predict_proba(sample)[0].tolist()
print({'prediction':pred,'label':meta['target_mapping'][str(pred)],'probabilities':proba})

## 3. Простейший мониторинг сдвига признаков

Сравним распределения обучающих и новых значений по стандартизованным средним. Это учебный сигнал для расследования, а не универсальный тест дрейфа.

In [ ]:
# Симулируем новые данные с изменённым масштабом одного признака
new_data=X_test.copy()
new_data['mean radius']=new_data['mean radius']*1.25
train_mean=X_train.mean(); train_std=X_train.std().replace(0,1)
shift=((new_data.mean()-train_mean).abs()/train_std).sort_values(ascending=False)
display(shift.head(10).rename('standardized_mean_shift').to_frame())
print('Признаки сдвиг > 0.5 SD:',shift[shift>0.5].index.tolist())

## Контрольные вопросы и вывод

1. Почему следует сохранять Pipeline целиком, а не одну модель?
2. Какие версии и метаданные нужны для воспроизводимости?
3. Чем отличается ошибка сервиса от деградации качества из-за дрейфа?
4. Почему средний сдвиг признака сам по себе не доказывает снижение качества?

**В отчёте:** перечень сохранённых артефактов и версии, пример инференса, найденные сигналы сдвига и ограничения мониторинга. В продакшне нужен контроль входной схемы, логирование, оценка качества после появления истинных меток, безопасное хранение данных и процедура обновления модели.